# Is the Lender Paid for the Risk? A Credit Pricing and Approval Audit

**Business question:** On a personal-loan book, does the interest rate charged by loan grade cover the default risk, and would a risk-based approval policy have improved expected profit?

**Dataset:** `credit_risk_dataset.csv` (32,581 loans, 12 columns). `loan_status = 1` means the loan defaulted. The file is anonymised and its source, currency, loan term and collection terms are not documented, so profit figures below rest on stated assumptions and are illustrative, not a real P&L.

**Approach:** (1) audit and clean the data, (2) measure default by segment with confidence intervals, (3) test whether each grade's rate covers its default rate via a break-even loss-given-default, (4) compare an applicant-only model with one that also sees the lender's grade and rate, (5) simulate approval cut-offs with sensitivity to the loss assumption, (6) state what the data cannot support.

In [ ]:
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.calibration import calibration_curve

sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = (10, 5.5)
DATA_PATH = Path('credit_risk_dataset.csv')   # edit if stored elsewhere, e.g. data/credit_risk_dataset.csv
OUT = Path('images_credit'); OUT.mkdir(exist_ok=True)
raw = pd.read_csv(DATA_PATH)
print(raw.shape); raw.head()

## 1. Data audit and cleaning
Checks run before any analysis. In the supplied file: 165 exact duplicate rows, 895 missing employment lengths, 3,116 missing interest rates, a maximum age of 144 and a maximum employment length of 123 years. Impossible values are set to missing (not deleted) so the loan still counts; missingness is kept as a flag because it may itself carry signal.

In [ ]:
print('Duplicates:', raw.duplicated().sum())
display(raw.isna().sum().rename('missing').to_frame().T)
df = raw.drop_duplicates().copy()
bad_age = df.person_age > 100
bad_emp = df.person_emp_length > 60
print('Impossible ages:', bad_age.sum(), '| impossible employment lengths:', bad_emp.sum())
df.loc[bad_age, 'person_age'] = np.nan
df.loc[bad_emp, 'person_emp_length'] = np.nan
df['rate_missing'] = df.loan_int_rate.isna().astype(int)
df['emp_missing'] = df.person_emp_length.isna().astype(int)
print('Rows after de-duplication:', len(df), '| default rate:', round(df.loan_status.mean(), 4))
print('Default rate when rate is missing vs present:'); print(df.groupby('rate_missing').loan_status.mean().round(4))

## 2. Engineered features
`income_band` and `lti_band` (loan-to-income) make segment comparisons readable; `rate_filled` fills a missing rate with the median for that grade so pricing maths can use every loan.

In [ ]:
df['income_band'] = pd.qcut(df.person_income, 5, labels=['Q1 lowest','Q2','Q3','Q4','Q5 highest'])
df['lti_band'] = pd.cut(df.loan_percent_income, [-.01, .1, .2, .3, 1], labels=['<10%','10-20%','20-30%','>30%'])
df['age_band'] = pd.cut(df.person_age, [19, 24, 29, 39, 100], labels=['20-24','25-29','30-39','40+'])
df['rate_filled'] = df.loan_int_rate.fillna(df.groupby('loan_grade').loan_int_rate.transform('median'))
def wilson(k, n, z=1.96):
    p = k/n; d = 1+z*z/n; c = p+z*z/(2*n); a = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))
    return (c-a)/d, (c+a)/d
def seg(col):
    t = df.groupby(col, observed=True).loan_status.agg(['size','sum']).rename(columns={'size':'loans','sum':'defaults'})
    t['default_rate'] = t.defaults/t.loans
    ci = [wilson(k, n) for k, n in zip(t.defaults, t.loans)]
    t['lo'] = [c[0] for c in ci]; t['hi'] = [c[1] for c in ci]
    return t
print('ready')

## 3. Where does default concentrate?
Default rate by segment with 95% Wilson intervals. Small groups (for example grade G, home ownership OTHER) have wide intervals, so read them with care.

In [ ]:
cols = ['loan_grade','loan_intent','person_home_ownership','lti_band','income_band','age_band','cb_person_default_on_file']
fig, axes = plt.subplots(3, 3, figsize=(17, 13)); axes = axes.ravel()
for ax, c in zip(axes, cols):
    t = seg(c); x = range(len(t))
    ax.bar(x, t.default_rate, color='#2b6cb0', yerr=[t.default_rate-t.lo, t.hi-t.default_rate], capsize=3)
    ax.axhline(df.loan_status.mean(), color='#c53030', ls='--', lw=1)
    ax.set_xticks(list(x)); ax.set_xticklabels([str(i) for i in t.index], rotation=35, ha='right', fontsize=8)
    ax.set_title(c); ax.set_ylim(0, 1.05)
for ax in axes[len(cols):]: ax.axis('off')
fig.suptitle('Default rate by segment (red line = portfolio average, bars = 95% CI)', y=1.0)
plt.tight_layout(); plt.savefig(OUT/'01_default_by_segment.png', dpi=160); plt.show()
display(seg('loan_grade').style.format({'default_rate':'{:.1%}','lo':'{:.1%}','hi':'{:.1%}'}))

### Are these differences statistically real?
Chi-square tests of independence between each categorical segment and default, with Cramer's V as effect size (a significant p-value on 32k rows can be a tiny effect, so effect size matters).

In [ ]:
rows = []
for c in cols:
    ct = pd.crosstab(df[c], df.loan_status)
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    v = np.sqrt(chi2/(ct.values.sum()*(min(ct.shape)-1)))
    rows.append({'variable':c, 'chi2':chi2, 'p_value':p, 'cramers_v':v})
assoc = pd.DataFrame(rows).sort_values('cramers_v', ascending=False).reset_index(drop=True)
display(assoc.style.format({'chi2':'{:.0f}','p_value':'{:.2e}','cramers_v':'{:.3f}'}))

## 4. Does the rate cover the risk? (the core analysis)
For a one-period loan, expected return per unit lent is approximately

$$E[R] = (1-p)\,r - p\,L$$

where $p$ is the default rate, $r$ the interest rate and $L$ the loss given default (share of principal not recovered). Setting $E[R]=0$ gives the **break-even loss given default**:

$$L^{*} = \frac{(1-p)\,r}{p}$$

If the true recovery loss is *higher* than $L^{*}$, that grade loses money. A low $L^{*}$ means the rate leaves almost no cushion. Loan term and recoveries are not in the data, so this treats each loan as a single period and is a screening tool, not an accounting result.

In [ ]:
g = df.groupby('loan_grade').agg(loans=('loan_status','size'), defaults=('loan_status','sum'), avg_rate=('rate_filled','mean'), avg_amount=('loan_amnt','mean'))
g['default_rate'] = g.defaults/g.loans
g['breakeven_LGD'] = (1-g.default_rate)*(g.avg_rate/100)/g.default_rate
g['rate_per_default_pt'] = g.avg_rate/(g.default_rate*100)
display(g.style.format({'avg_amount':'{:,.0f}','avg_rate':'{:.2f}','default_rate':'{:.1%}','breakeven_LGD':'{:.1%}','rate_per_default_pt':'{:.2f}'}))
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
ax[0].plot(g.index, g.avg_rate, marker='o', label='Average interest rate (%)')
ax[0].plot(g.index, g.default_rate*100, marker='s', label='Default rate (%)'); ax[0].legend()
ax[0].set_title('Rate rises steadily, default jumps from grade D'); ax[0].set_xlabel('Loan grade')
ax[1].bar(g.index, g.breakeven_LGD*100, color='#2f855a'); ax[1].axhline(60, color='#c53030', ls='--', label='Illustrative 60% loss')
ax[1].set_title('Break-even loss given default (%)'); ax[1].legend(); ax[1].set_xlabel('Loan grade')
plt.tight_layout(); plt.savefig(OUT/'02_pricing_adequacy.png', dpi=160); plt.show()

**How to read this:** a break-even loss of, say, 10% means the grade only makes money if the lender recovers 90% of principal on defaulted loans, which is unrealistic for unsecured credit. Read the grades against the 60% line: any grade whose bar sits below it is mispriced under that loss assumption. Report the exact grades from the table above.

## 5. Predictive model: applicant-only versus lender-assigned inputs
Two feature sets, compared on a stratified 25% hold-out:
- **Applicant-only:** age, income, employment length, loan amount, loan-to-income, credit-history length, home ownership, loan intent, prior default flag.
- **With grade and rate:** adds `loan_grade` and `loan_int_rate`, which the lender sets *after* assessing the applicant. These may already encode the lender's own risk view, so a gain here shows the lender's score is informative, not that an applicant-only model is weak.

Metrics: ROC-AUC (ranking), Brier score (probability accuracy, lower is better) and a calibration curve. A logistic regression is the transparent baseline; gradient boosting shows how much non-linearity adds.

In [ ]:
num_a = ['person_age','person_income','person_emp_length','loan_amnt','loan_percent_income','cb_person_cred_hist_length','emp_missing']
cat_a = ['person_home_ownership','loan_intent','cb_person_default_on_file']
num_b = num_a + ['loan_int_rate','rate_missing']; cat_b = cat_a + ['loan_grade']
def pipe(num, cat, model):
    pre = ColumnTransformer([('n', Pipeline([('i', SimpleImputer(strategy='median')), ('s', StandardScaler())]), num),
                             ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
    return Pipeline([('p', pre), ('m', model)])
train, test = train_test_split(df, test_size=0.25, stratify=df.loan_status, random_state=42)
specs = {('Applicant-only','Logistic'):(num_a,cat_a,LogisticRegression(max_iter=2000)),
         ('Applicant-only','Boosting'):(num_a,cat_a,HistGradientBoostingClassifier(random_state=42)),
         ('With grade + rate','Logistic'):(num_b,cat_b,LogisticRegression(max_iter=2000)),
         ('With grade + rate','Boosting'):(num_b,cat_b,HistGradientBoostingClassifier(random_state=42))}
preds, rows = {}, []
for (fs, mn), (n, c, m) in specs.items():
    p = pipe(n, c, m).fit(train[n+c], train.loan_status); pr = p.predict_proba(test[n+c])[:, 1]; preds[(fs, mn)] = pr
    rows.append({'features':fs, 'model':mn, 'ROC-AUC':roc_auc_score(test.loan_status, pr), 'Brier':brier_score_loss(test.loan_status, pr)})
results = pd.DataFrame(rows); display(results.style.format({'ROC-AUC':'{:.3f}','Brier':'{:.3f}'}))
fig, ax = plt.subplots(figsize=(6.5, 6)); ax.plot([0,1],[0,1],'k--',lw=1,label='Perfect calibration')
for k, pr in preds.items():
    fx, fy = calibration_curve(test.loan_status, pr, n_bins=10, strategy='quantile'); ax.plot(fy, fx, marker='o', label=' / '.join(k))
ax.set(xlabel='Predicted default probability', ylabel='Observed default rate', title='Calibration on hold-out'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'03_calibration.png', dpi=160); plt.show()

## 6. Approval-policy simulation
Using the **applicant-only boosting model** (it does not rely on the lender's own grade), we sweep an approval cut-off on the hold-out set: approve a loan only if its predicted default probability is below the cut-off.

Assumptions, all stated and changeable: profit on a repaid loan = `loan_amnt x rate`; loss on a default = `loan_amnt x LGD`; one period, no funding cost, no fees. Because LGD is unknown, the whole analysis is repeated for LGD of 40%, 60% and 80%. The cut-off with the best profit is chosen on the hold-out set, so the figure is optimistic; treat it as an indicator, not a forecast.

In [ ]:
pd_hat = preds[('Applicant-only','Boosting')]
t = test.assign(pd_hat=pd_hat)
t['gain'] = t.loan_amnt * t.rate_filled / 100
cuts = np.linspace(0.05, 1.0, 96)
fig, ax = plt.subplots(figsize=(10, 5.5)); summ = []
for lgd in [0.4, 0.6, 0.8]:
    t['pnl'] = np.where(t.loan_status == 1, -t.loan_amnt*lgd, t.gain)
    curve = [t.loc[t.pd_hat <= c, 'pnl'].sum() for c in cuts]
    best = int(np.argmax(curve)); ax.plot(cuts, curve, label=f'LGD {lgd:.0%}')
    ax.scatter(cuts[best], curve[best], zorder=5)
    app = t[t.pd_hat <= cuts[best]]
    summ.append({'LGD':f'{lgd:.0%}','approve_all_profit':t.pnl.sum(),'best_cutoff':round(cuts[best],2),'best_profit':curve[best],
                 'approval_rate':len(app)/len(t),'default_rate_if_approved':app.loan_status.mean()})
ax.axhline(0, color='k', lw=.8); ax.set(xlabel='Approve if predicted default probability <=', ylabel='Total hold-out profit (currency units)', title='Profit versus approval cut-off'); ax.legend()
plt.tight_layout(); plt.savefig(OUT/'04_policy_simulation.png', dpi=160); plt.show()
display(pd.DataFrame(summ).style.format({'approve_all_profit':'{:,.0f}','best_profit':'{:,.0f}','approval_rate':'{:.1%}','default_rate_if_approved':'{:.1%}'}))

## 7. Fairness and use caution
Age is in the model for a descriptive comparison only. Using age to decide credit can be unlawful in many jurisdictions, and home ownership and income can act as proxies for protected traits. A production model would need legal review, and the check below shows whether predictions differ sharply by age band.

In [ ]:
t['approved'] = t.pd_hat <= 0.30
fair = t.groupby('age_band', observed=True).agg(loans=('approved','size'), approval_rate=('approved','mean'), actual_default=('loan_status','mean'), mean_predicted=('pd_hat','mean'))
display(fair.style.format({'approval_rate':'{:.1%}','actual_default':'{:.1%}','mean_predicted':'{:.1%}'}))

## 8. Findings (complete from your run)
Fill each line with numbers from the tables above. Claims must match the output.

1. **Pricing:** which grades have a break-even LGD below 60%, and how much of the loan book do they hold (`g.loans`)?
2. **Risk concentration:** which segments sit outside the 95% interval around the portfolio default rate, and which have the largest Cramer's V?
3. **Model value:** AUC and Brier for applicant-only versus with grade and rate; how much does the lender's own grade add?
4. **Policy:** best cut-off, approval rate and profit change versus approve-all, at each LGD. Does the conclusion hold across all three?
5. **Fairness:** do approval rates differ materially by age band?

## 9. Limitations
- Source, currency, term, collateral and recovery are undocumented; the profit model is a one-period simplification with assumed LGD.
- Default rates for grades D to G (about 59% to 98% in this file) are far higher than typical real lending books, which suggests the data is synthetic or heavily sampled. Use it for method, not for real-world benchmarks.
- Rows are loans, not unique people; repeat borrowers cannot be identified.
- Cut-off chosen on the hold-out set is optimistic; a separate validation split or cross-validation would be needed for a deployable policy.
- Observational data shows association, not causation.